In [29]:
import argparse
import os
import pathlib
import sys

import numpy as np
import pandas as pd
from image_analysis_3D.featurization_utils.feature_writing_utils import (
    format_morphology_feature_name,
)
from image_analysis_3D.featurization_utils.neighbors_utils import (
    classify_cells_into_shells,
    euclidean_distance_from_centroid,
    mahalanobis_distance_from_centroid,
)
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from tqdm import tqdm

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(), root_dir
)

In [2]:
if not in_notebook:
    args = parse_args()
    well_fov = args["well_fov"]
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0014_T1"
    well_fov = "C4-2"
    image_based_profiles_subparent_name = "image_based_profiles"

In [3]:
def centroid_within_bbox_detection(
    centroid: tuple,
    bbox: tuple,
) -> bool:
    """
    Check if the centroid is within the bbox

    Parameters
    ----------
    centroid : tuple
        Centroid of the object in the order of (z, y, x)
        Order of the centroid is important
    bbox : tuple
        Where the bbox is in the order of (z_min, y_min, x_min, z_max, y_max, x_max)
        Order of the bbox is important

    Returns
    -------
    bool
        True if the centroid is within the bbox, False otherwise
    """
    z_min, y_min, x_min, z_max, y_max, x_max = bbox
    z, y, x = centroid
    # check if the centroid is within the bbox
    if (
        z >= z_min
        and z <= z_max
        and y >= y_min
        and y <= y_max
        and x >= x_min
        and x <= x_max
    ):
        return True
    else:
        return False

### Pathing

In [4]:
# input paths
sc_profile_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/sc_profiles_{well_fov}.parquet"
).resolve(strict=True)
organoid_profile_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/organoid_profiles_{well_fov}.parquet"
).resolve(strict=True)
nucleocentric_profile_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/nucleocentric_profiles_{well_fov}.parquet"
).resolve(strict=True)
# output paths
sc_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/sc_profiles_{well_fov}_related.parquet"
).resolve()
organoid_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/organoid_profiles_{well_fov}_related.parquet"
).resolve()
nucleocentric_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles/{well_fov}/nucleocentric_profiles_{well_fov}_related.parquet"
).resolve()
sc_profile_output_path.parent.mkdir(parents=True, exist_ok=True)

In [5]:
sc_profile_df = pd.read_parquet(sc_profile_path)
nucleocentric_df = pd.read_parquet(nucleocentric_profile_path)
organoid_profile_df = pd.read_parquet(organoid_profile_path)
print(f"Single-cell profile shape: {sc_profile_df.shape}")
print(f"Nucleocentric profile shape: {nucleocentric_df.shape}")
print(f"Organoid profile shape: {organoid_profile_df.shape}")

Single-cell profile shape: (41, 10026)
Nucleocentric profile shape: (41, 3074)
Organoid profile shape: (1, 3337)


In [6]:
# initialize the parent organoid column
sc_profile_df.insert(2, "ParentOrganoid", -1)

In [44]:
x_y_z_sc_colnames = [
    x
    for x in sc_profile_df.columns
    if "area" in x.lower() and "center" in x.lower() and "nuclei" in x.lower()
]
x_y_z_sc_colnames

['Nuclei_NoChannel_AreaSizeShape_CenterX',
 'Nuclei_NoChannel_AreaSizeShape_CenterY',
 'Nuclei_NoChannel_AreaSizeShape_CenterZ',
 'Nuclei_DNA_AreaSizeShape_CenterX',
 'Nuclei_DNA_AreaSizeShape_CenterY',
 'Nuclei_DNA_AreaSizeShape_CenterZ']

In [8]:
organoid_bbox_colnames = [
    x
    for x in organoid_profile_df.columns
    if "area" in x.lower() and ("min" in x.lower() or "max" in x.lower())
]
organoid_bbox_colnames = sorted(organoid_bbox_colnames)

In [9]:
sc_centroids = sc_profile_df[
    x_y_z_sc_colnames
].values  # alphabetically sorted to be in the order of x,y,z

In [10]:
# Initialize parent_organoid to -1
sc_profile_df["ParentOrganoid"] = -1

# Extract single-cell centroids as numpy array for faster access
sc_centroids = sc_profile_df[x_y_z_sc_colnames].values  # (N_cells, 3) array
# reshape the centroids to be in z,y,x order for easier comparison with bbox
sc_centroids = sc_centroids[:, [2, 1, 0]]  # reorder to z,y,x

# Loop through organoids with progress bar
for organoid_index, organoid_row in tqdm(
    organoid_profile_df.iterrows(),
    total=len(organoid_profile_df),
    desc="Assigning cells to organoids",
):
    # Get organoid bbox
    organoid_bbox = (
        organoid_row[organoid_bbox_colnames[5]],  # z_min
        organoid_row[organoid_bbox_colnames[4]],  # y_min
        organoid_row[organoid_bbox_colnames[3]],  # x_min
        organoid_row[organoid_bbox_colnames[2]],  # z_max
        organoid_row[organoid_bbox_colnames[1]],  # y_max
        organoid_row[organoid_bbox_colnames[0]],  # x_max
    )

    z_min, y_min, x_min, z_max, y_max, x_max = organoid_bbox

    # Vectorized bbox check - much faster!
    mask = (
        (sc_centroids[:, 0] >= z_min)  # z
        & (sc_centroids[:, 0] <= z_max)  # z
        & (sc_centroids[:, 1] >= y_min)
        & (sc_centroids[:, 1] <= y_max)
        & (sc_centroids[:, 2] >= x_min)
        & (sc_centroids[:, 2] <= x_max)
    )

    # Only assign if cell doesn't already have a parent
    unassigned_mask = sc_profile_df["ParentOrganoid"] == -1
    final_mask = mask & unassigned_mask

    # Assign parent organoid to matching cells
    sc_profile_df.loc[final_mask, "ParentOrganoid"] = organoid_row["object_id"]

print(f"Assigned {(sc_profile_df['ParentOrganoid'] != -1).sum()} cells to organoids")
print(f"Unassigned cells: {(sc_profile_df['ParentOrganoid'] == -1).sum()}")

Assigning cells to organoids: 100%|██████████| 1/1 [00:00<00:00, 623.32it/s]

Assigned 41 cells to organoids
Unassigned cells: 0


### Add single-cell counts for each organoid

In [11]:
organoid_sc_counts = (
    sc_profile_df["ParentOrganoid"]
    .value_counts()
    .to_frame(name="SingleCellCount")
    .reset_index()
)
# merge the organoid profile with the single-cell counts
organoid_profile_df = pd.merge(
    organoid_profile_df,
    organoid_sc_counts,
    left_on="object_id",
    right_on="ParentOrganoid",
    how="left",
).drop(columns=["ParentOrganoid"])
sc_count = organoid_profile_df.pop("SingleCellCount")
organoid_profile_df.insert(2, "SingleCellCount", sc_count)

Even if the file is empty we still want to add it to the final dataframe dictionary so that we can merge on the same columns later.
This will help with file-based checking and merging.


In [12]:
# replace NaN with 0 for organoids that have no assigned cells
organoid_profile_df["SingleCellCount"] = (
    organoid_profile_df["SingleCellCount"].fillna(0).astype(int)
)
organoid_profile_df.head()

,object_id,image_set,SingleCellCount,Organoid_NoChannel_AreaSizeShape_Volume,Organoid_NoChannel_AreaSizeShape_CenterX,Organoid_NoChannel_AreaSizeShape_CenterY,Organoid_NoChannel_AreaSizeShape_CenterZ,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_MinX,Organoid_NoChannel_AreaSizeShape_MaxX,...,Organoid_Mito_Texture_DifferenceEntropy-256-3,Organoid_Mito_Texture_DifferenceVariance-256-3,Organoid_Mito_Texture_Entropy-256-3,Organoid_Mito_Texture_InformationMeasureOfCorrelation1-256-3,Organoid_Mito_Texture_InformationMeasureOfCorrelation2-256-3,Organoid_Mito_Texture_InverseDifferenceMoment-256-3,Organoid_Mito_Texture_SumAverage-256-3,Organoid_Mito_Texture_SumEntropy-256-3,Organoid_Mito_Texture_SumVariance-256-3,Organoid_Mito_Texture_Variance-256-3
0,1,C4-2,41,18671184.0,671.803338,563.720671,14.722934,30119463.0,229,1150,...,1.307112,0.002466,2.531704,-0.486536,0.89619,0.839941,7.675781,1.927611,193.311708,49.2295


In [13]:
if organoid_profile_df.empty:
    # add a row with 0 values
    organoid_profile_df.loc[len(organoid_profile_df)] = [0] * len(
        organoid_profile_df.columns
    )
    organoid_profile_df["image_set"] = well_fov

In [14]:
print(f"Single-cell profile shape: {sc_profile_df.shape}")

Single-cell profile shape: (41, 10027)


In [15]:
if sc_profile_df.empty:
    # add a row with Na values
    sc_profile_df.loc[len(sc_profile_df)] = [None] * len(sc_profile_df.columns)
    sc_profile_df["image_set"] = well_fov

In [16]:
# add the parent organoid to nucleocentric features
nucleocentric_df = pd.merge(
    nucleocentric_df,
    sc_profile_df[["object_id", "image_set", "ParentOrganoid"]],
    on=["object_id", "image_set"],
    how="left",
)

## Get single cell and organoid relationships and spatial distributions

In [17]:
x_y_z_organoid_centroid_colnames = [
    x
    for x in organoid_profile_df.columns
    if "area" in x.lower() and "center" in x.lower()
]
x_y_z_organoid_bbox_colnames = [
    x
    for x in organoid_profile_df.columns
    if "area" in x.lower() and ("min" in x.lower() or "max" in x.lower())
]

In [37]:
results = []

# get the organoid id and the single-cells for each

organoid_ids = organoid_profile_df["object_id"]

# organoid_id = organoid_ids[0]

for organoid_id in organoid_ids:
    organoid_centroid = (
        organoid_profile_df.loc[
            organoid_profile_df["object_id"] == organoid_id,
            x_y_z_organoid_centroid_colnames,
        ]
        .apply(pd.to_numeric, errors="coerce")
        .iloc[0]
        .to_numpy(dtype=float)
    )
    organoid_bbox = organoid_profile_df.loc[
        organoid_profile_df["object_id"] == organoid_id, x_y_z_organoid_bbox_colnames
    ].values[0]
    single_cells_in_organoid = sc_profile_df[
        sc_profile_df["ParentOrganoid"] == organoid_id
    ]
    if single_cells_in_organoid.empty:
        print(f"No single cells assigned to organoid {organoid_id}")
        continue

    single_cells_centroids = (
        single_cells_in_organoid[x_y_z_sc_colnames]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=float)
    )

    valid_rows = ~pd.isna(single_cells_centroids).any(axis=1)
    single_cells_centroids = single_cells_centroids[valid_rows]
    single_cells_in_organoid = single_cells_in_organoid.loc[valid_rows]

    if single_cells_centroids.shape[0] == 0:
        continue

    # convert to a dict with the key being the object_id
    # rename the centroids to z,y.x
    single_cells_centroids_dict = {
        "object_id": single_cells_in_organoid["object_id"].to_numpy(),
        "z": single_cells_in_organoid[x_y_z_sc_colnames[2]].to_numpy(dtype=float),
        "y": single_cells_in_organoid[x_y_z_sc_colnames[1]].to_numpy(dtype=float),
        "x": single_cells_in_organoid[x_y_z_sc_colnames[0]].to_numpy(dtype=float),
    }

    euclidean_distance = euclidean_distance_from_centroid(
        single_cells_centroids, organoid_centroid
    )
    mahalanobis_distance = mahalanobis_distance_from_centroid(
        single_cells_centroids, organoid_centroid
    )
    shell_classification, centroid = classify_cells_into_shells(
        coords=single_cells_centroids_dict,
        n_shells=4,
        method="mahalanobis",
        min_cells_per_shell=3,
        centroid=organoid_centroid,
    )

    shell_classification_df = pd.DataFrame(shell_classification)
    shell_classification_df["ParentOrganoid"] = organoid_id
    results.append(shell_classification_df)

ValueError: operands could not be broadcast together with shapes (41,6) (3,) 

In [41]:
import numpy

numpy.sqrt(numpy.sum((single_cells_centroids - organoid_centroid) ** 2, axis=1))

ValueError: operands could not be broadcast together with shapes (41,6) (3,) 

In [42]:
single_cells_centroids

array([[ 504.41373441,  254.19931211,    4.22116125,  504.41373441,
         254.19931211,    4.22116125],
       [ 400.76396032,  695.12762534,    4.47368858,  400.76396032,
         695.12762534,    4.47368858],
       [ 573.77017079,  885.10738591,    3.44678918,  573.77017079,
         885.10738591,    3.44678918],
       [ 742.89692332,  386.98420212,    4.94334387,  742.89692332,
         386.98420212,    4.94334387],
       [ 469.86539389,  554.11424788,    7.75145544,  469.86539389,
         554.11424788,    7.75145544],
       [ 412.5576205 ,  537.38862353,   16.94884933,  412.5576205 ,
         537.38862353,   16.94884933],
       [1104.63058472,  674.21981338,    3.96876109, 1104.63058472,
         674.21981338,    3.96876109],
       [ 565.25505663,  803.02543691,    5.60561325,  565.25505663,
         803.02543691,    5.60561325],
       [ 698.32603604,  801.51626962,    7.76024412,  698.32603604,
         801.51626962,    7.76024412],
       [ 714.99633621,  223.43896336,

In [38]:
organoid_centroid

array([671.80333797, 563.72067069,  14.72293444])

In [39]:
single_cells_centroids

array([[ 504.41373441,  254.19931211,    4.22116125,  504.41373441,
         254.19931211,    4.22116125],
       [ 400.76396032,  695.12762534,    4.47368858,  400.76396032,
         695.12762534,    4.47368858],
       [ 573.77017079,  885.10738591,    3.44678918,  573.77017079,
         885.10738591,    3.44678918],
       [ 742.89692332,  386.98420212,    4.94334387,  742.89692332,
         386.98420212,    4.94334387],
       [ 469.86539389,  554.11424788,    7.75145544,  469.86539389,
         554.11424788,    7.75145544],
       [ 412.5576205 ,  537.38862353,   16.94884933,  412.5576205 ,
         537.38862353,   16.94884933],
       [1104.63058472,  674.21981338,    3.96876109, 1104.63058472,
         674.21981338,    3.96876109],
       [ 565.25505663,  803.02543691,    5.60561325,  565.25505663,
         803.02543691,    5.60561325],
       [ 698.32603604,  801.51626962,    7.76024412,  698.32603604,
         801.51626962,    7.76024412],
       [ 714.99633621,  223.43896336,

In [ ]:
if results:
    df = pd.concat([pd.DataFrame(r) for r in results], ignore_index=True)

else:
    df = pd.DataFrame(columns=["object_id", "ParentOrganoid"])

# rename the columns

df.rename(
    columns={
        col: format_morphology_feature_name(
            compartment="Nuclei",
            feature_type="Neighbors",
            channel="NoChannel",
            measurement=col,
        )
        for col in df.columns
        if col not in ["object_id", "ParentOrganoid"]
    },
    inplace=True,
)

In [ ]:
# concat the shell classification with the single cell profile df to get the full single cell profile with the shell classification and the parent organoid id
sc_profile_with_shells_df = pd.merge(
    sc_profile_df,
    df,
    left_on=["object_id", "ParentOrganoid"],
    right_on=["object_id", "ParentOrganoid"],
    how="left",
)

### Save the profiles

In [ ]:
organoid_profile_df.to_parquet(organoid_profile_output_path, index=False)
organoid_profile_df.head()

,object_id,image_set,SingleCellCount,Organoid_NoChannel_AreaSizeShape_Volume,Organoid_NoChannel_AreaSizeShape_CenterX,Organoid_NoChannel_AreaSizeShape_CenterY,Organoid_NoChannel_AreaSizeShape_CenterZ,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_MinX,Organoid_NoChannel_AreaSizeShape_MaxX,...,Organoid_Mito_Texture_DifferenceEntropy-256-3,Organoid_Mito_Texture_DifferenceVariance-256-3,Organoid_Mito_Texture_Entropy-256-3,Organoid_Mito_Texture_InformationMeasureOfCorrelation1-256-3,Organoid_Mito_Texture_InformationMeasureOfCorrelation2-256-3,Organoid_Mito_Texture_InverseDifferenceMoment-256-3,Organoid_Mito_Texture_SumAverage-256-3,Organoid_Mito_Texture_SumEntropy-256-3,Organoid_Mito_Texture_SumVariance-256-3,Organoid_Mito_Texture_Variance-256-3
0,1,C4-1,55,23163957.0,705.592170,951.984088,23.421290,50936760.0,279,1091,...,1.372933,0.002564,2.284438,-0.480396,0.873935,0.823807,16.806490,1.883787,1019.969193,260.433347
1,2,C4-1,0,1090.0,822.920183,322.218349,12.352294,4675.0,814,831,...,0.000345,0.003891,0.000384,-0.310887,0.011695,0.999984,0.003263,0.000362,0.721209,0.252476


In [ ]:
sc_profile_with_shells_df.to_parquet(sc_profile_output_path, index=False)
sc_profile_with_shells_df.head()

,object_id,image_set,ParentOrganoid,Nuclei_NoChannel_AreaSizeShape_Volume,Nuclei_NoChannel_AreaSizeShape_CenterX,Nuclei_NoChannel_AreaSizeShape_CenterY,Nuclei_NoChannel_AreaSizeShape_CenterZ,Nuclei_NoChannel_AreaSizeShape_BboxVolume,Nuclei_NoChannel_AreaSizeShape_MinX,Nuclei_NoChannel_AreaSizeShape_MaxX,...,Cytoplasm_ER_Texture_InverseDifferenceMoment-256-3,Cytoplasm_ER_Texture_SumAverage-256-3,Cytoplasm_ER_Texture_SumEntropy-256-3,Cytoplasm_ER_Texture_SumVariance-256-3,Cytoplasm_ER_Texture_Variance-256-3,Nuclei_NoChannel_Neighbors_ShellAssignments,Nuclei_NoChannel_Neighbors_DistancesFromCenter,Nuclei_NoChannel_Neighbors_DistancesFromExterior,Nuclei_NoChannel_Neighbors_NormalizedDistancesFromCenter,Nuclei_NoChannel_Neighbors_ShellsUsed
0,1,C4-1,1,37952.0,570.619098,889.471912,4.401138,53088.0,533,612,...,0.997082,2.074524,0.083098,599.464738,156.159871,3,1.901460,0.629340,0.751328,4
1,2,C4-1,1,45730.0,630.845987,980.079357,4.680691,80442.0,576,685,...,0.998868,0.334876,0.024220,72.396668,20.269828,2,1.729632,0.801169,0.683433,4
2,3,C4-1,1,32792.0,513.989388,1233.826391,3.298061,55692.0,471,562,...,0.998523,0.447109,0.029871,102.550110,28.117043,3,2.250824,0.279976,0.889372,4
3,4,C4-1,1,78837.0,805.883583,657.817573,9.887451,139956.0,752,859,...,0.996896,1.930529,0.072111,599.771707,158.815564,2,1.497050,1.033751,0.591532,4
4,5,C4-1,1,106256.0,837.627917,1101.988151,12.167463,171360.0,788,890,...,0.998287,0.576406,0.040535,118.019967,32.303203,2,1.754080,0.776720,0.693093,4


In [ ]:
nucleocentric_df.to_parquet(nucleocentric_profile_output_path, index=False)
nucleocentric_df.head()

,object_id,image_set,Nucleocentric_ER_CHAMMI75_Feature0,Nucleocentric_ER_CHAMMI75_Feature1,Nucleocentric_ER_CHAMMI75_Feature10,Nucleocentric_ER_CHAMMI75_Feature100,Nucleocentric_ER_CHAMMI75_Feature101,Nucleocentric_ER_CHAMMI75_Feature102,Nucleocentric_ER_CHAMMI75_Feature103,Nucleocentric_ER_CHAMMI75_Feature104,...,Nucleocentric_DNA_SAMMed3D_Feature91,Nucleocentric_DNA_SAMMed3D_Feature92,Nucleocentric_DNA_SAMMed3D_Feature93,Nucleocentric_DNA_SAMMed3D_Feature94,Nucleocentric_DNA_SAMMed3D_Feature95,Nucleocentric_DNA_SAMMed3D_Feature96,Nucleocentric_DNA_SAMMed3D_Feature97,Nucleocentric_DNA_SAMMed3D_Feature98,Nucleocentric_DNA_SAMMed3D_Feature99,ParentOrganoid
0,1,C4-1,-2.077652,-3.769540,7.392308,-2.904450,0.304836,-0.125571,3.759429,1.653862,...,-0.063573,0.012375,-0.010451,0.018821,-0.035349,-0.072266,0.225062,0.375727,0.230372,1
1,2,C4-1,1.937729,-3.106619,3.096841,2.169847,-0.647294,-3.975332,2.363536,-1.013883,...,-0.074480,-0.133599,-0.010789,0.014345,-0.093833,0.208869,0.276891,0.316324,0.034449,1
2,3,C4-1,0.399118,-2.973526,2.263088,2.779027,0.539426,-0.112798,4.970295,-2.148763,...,-0.088939,0.062766,-0.010760,0.023554,0.038052,0.025307,0.201645,0.315295,0.227340,1
3,4,C4-1,-2.808135,-5.294311,0.994752,2.960989,-2.145890,1.459159,6.289212,-0.280200,...,-0.036521,0.055151,-0.010306,0.020021,-0.006468,-0.026702,0.222935,0.391188,0.207675,1
4,5,C4-1,2.550292,-0.210102,2.818901,-1.220781,-2.845022,-0.549444,4.041600,-1.673978,...,-0.062398,0.051756,-0.010719,0.003777,-0.037100,-0.121255,0.226659,0.402324,0.156472,1
